# Bagging e Random Forests

**Objetivo:** ver a acurácia crescer e estabilizar com o número de árvores, comparar a floresta com uma árvore isolada e ler a importância das variáveis.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

dados = load_breast_cancer()
X, y = dados.data, dados.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)
print("treino:", X_tr.shape, "| preditores:", X.shape[1])

## 1. Uma árvore × uma floresta

A árvore isolada tem variância alta; a floresta (bagging + aleatoriedade nas variáveis) estabiliza.

In [ ]:
arvore = DecisionTreeClassifier(random_state=SEMENTE).fit(X_tr, y_tr)
floresta = RandomForestClassifier(n_estimators=300, random_state=SEMENTE).fit(X_tr, y_tr)
print("acuracia da arvore isolada:", round(arvore.score(X_te, y_te), 3))
print("acuracia da floresta (300):", round(floresta.score(X_te, y_te), 3))

## 2. Acurácia × número de árvores

A acurácia sobe rápido com as primeiras árvores e depois estabiliza — acrescentar árvores **não** causa overfitting (só custa tempo).

In [ ]:
numeros = [1, 2, 5, 10, 25, 50, 100, 200, 400]
acuracias = []
for n in numeros:
    modelo = RandomForestClassifier(n_estimators=n, random_state=SEMENTE)
    modelo.fit(X_tr, y_tr)
    acuracias.append(modelo.score(X_te, y_te))
    print("arvores", str(n).rjust(3), "-> acuracia", round(acuracias[-1], 3))

figura = go.Figure(go.Scatter(x=numeros, y=acuracias, mode="lines+markers",
                              line=dict(color=AZUL)))
figura.update_layout(title="Acuracia vs numero de arvores (bagging estabiliza)",
                     xaxis_title="n_estimators", yaxis_title="acuracia", height=340,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. Importância das variáveis

A floresta entrega, de graça, um ranking de quais preditores mais reduzem a impureza. Mostramos os dez maiores.

In [ ]:
importancias = floresta.feature_importances_
ordem = np.argsort(importancias)[::-1][:10]
nomes_top = [dados.feature_names[j] for j in ordem]
valores_top = importancias[ordem]
for nome, val in zip(nomes_top, valores_top):
    print(nome.ljust(24), round(val, 3))

figura = go.Figure(go.Bar(x=valores_top[::-1], y=nomes_top[::-1], orientation="h",
                          marker_color=VERDE))
figura.update_layout(title="Importancia das variaveis (top 10)",
                     xaxis_title="importancia", height=380,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercício

Compare a acurácia da árvore isolada com a da floresta de 300 árvores. O ganho vem de reduzir viés ou variância? Justifique.

<details><summary>Ver resposta</summary>

Vem de reduzir **variância**. A árvore isolada e a floresta têm o mesmo tipo de modelo de base (árvores), com viés parecido; o que a floresta faz é tirar a **média** de muitas árvores decorrelacionadas, o que reduz a variância do conjunto — daí a acurácia mais alta e estável. Não muda a natureza do viés, estabiliza a variância.

</details>